In [1]:
import os
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Resolve absolute paths to local Silver Delta table directories
PROJECT_ROOT = os.path.dirname(os.getcwd())


GOLD_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "gold_security_alerts")

# Initialize SparkSession with Delta support
builder = (
    SparkSession.builder.appName("Query_Gold")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

:: loading settings :: url = jar:file:/Users/intuitivecode/Code/deltastreams/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/intuitivecode/.ivy2/cache
The jars for the packages stored in: /Users/intuitivecode/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e3547ce7-9c44-4967-9967-b9b0862450d4;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 85ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default    

In [2]:
# Read the materialized Gold summary table
gold_alerts_df = spark.read.format("delta").load(GOLD_PATH)

print(f"Total Operational Security Alerts Triggered: {gold_alerts_df.count()}")
gold_alerts_df.orderBy(col("_alert_triggered_at").desc()).show(10, truncate=False)

26/10/18 13:30:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Total Operational Security Alerts Triggered: 6
+-------------------+-------------------+-------------+--------+-------------+-----------------------+
|alert_window_start |alert_window_end   |attacker_ip  |protocol|request_count|_alert_triggered_at    |
+-------------------+-------------------+-------------+--------+-------------+-----------------------+
|2026-10-18 13:27:30|2026-10-18 13:28:30|192.168.1.100|TCP     |5            |2026-10-18 13:29:08.021|
|2026-10-17 17:46:30|2026-10-17 17:47:30|192.168.1.100|TCP     |34           |2026-10-18 13:29:08.021|
|2026-10-18 13:28:00|2026-10-18 13:29:00|192.168.1.100|TCP     |5            |2026-10-18 13:29:08.021|
|2026-10-17 17:47:00|2026-10-17 17:48:00|192.168.1.100|TCP     |13           |2026-10-18 13:29:08.021|
|2026-10-17 17:46:00|2026-10-17 17:47:00|192.168.1.100|TCP     |36           |2026-10-18 13:29:08.021|
|2026-10-17 17:45:30|2026-10-17 17:46:30|192.168.1.100|TCP     |15           |2026-10-18 13:29:08.021|
+-------------------+-----

26/10/18 13:31:04 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
